<a href="https://colab.research.google.com/github/miso-20/statistic_programing_2/blob/main/6_4)_pandas_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 인구 데이터 실습
- 2024년 11월 인구 자료를 이용하여 `merge`, `groupby`, Simple regression을 적용한다.
- 행정구역 코드로 시도, 시군구, 읍면동을 연결하고, `si`와 `sgg`를 구분한다.
- 마지막으로 읍면동별 총인구와 고령인구를 이용하여 단순선형회귀를 계산한다.

## 1. 데이터 구조
- 데이터는 행정구역 $\times$ 연령구분으로 구성되어있으며, 하나의 행정구역에 전체 인구(계)와 5세 단위 연령별 인구가 각각 저장되어 있다. 변수는 지역코드, 지역명, 연령코드, 인구로 구성되어 있다.
- 행정구역은 일반적으로 시도, 시군구, 읍면동으로 구분된다. 하지만 일부 지역에는 시 아래에 일반구가 존재한다. 행정구역 코드는 기본적으로 `2자리(시도), 5자리(시군구), 10자리(읍면동)`로 구분된다. 다만 일부 지역에서는 시와 일반구가 모두 5자리 코드를 사용한다.
- 예를 들어, 서울특별시는 `11(서울특별시), 11110(종로구), 1111051500(청운효자동)`으로 연결되있지만, 경기도 수원시의 경우, `41(경기도), 41110(수원시), 41111(장안구), 4111156000(파장동)`으로 연결되어 있다.
- 따라서, 지역에 따라 5자리 지역코드가 상위 시를 의미하거나, 일반구를 의미할 수 있다.

## 2. CSV 데이터 읽기
- 원본에는 총인구(`계`)와 연령별 인구가 함께 저장되어 있다.
- 지역코드는 앞자리 `0`이 유지되도록 문자열로 읽는다.

In [ ]:
import pandas as pd
import numpy as np

tmp_dat = pd.read_csv("C:/Users/98mij/OneDrive/바탕 화면/김미소/통계프로그래밍2/총인구_2024.csv", encoding = "cp949")

tmp_dat = tmp_dat[["[A]행정구역(동읍면)별", "행정구역(동읍면)별", "[B]5세별", "2024.11 월"]]
tmp_dat.columns = ["지역코드", "지역명", "연령코드", "인구"]

tmp_dat.head()

,지역코드,지역명,연령코드,인구
0,0,전국,0,51226757
1,0,전국,5,1272796
2,0,전국,10,1879586
3,0,전국,15,2325120
4,0,전국,20,2295875


In [ ]:
import pandas as pd
import numpy as np

tmp_dat = pd.read_csv("C:/Users/98mij/OneDrive/바탕 화면/김미소/통계프로그래밍2/총인구_2024.csv", encoding = "cp949", dtype = {"[A]행정구역(동읍면)별": str})
# dtype = {"[A]행정구역(동읍면)별": str})  :  데이터 type 인식시켜줌 -> 여기선 반드시 들어가야 됨

tmp_dat = tmp_dat[["[A]행정구역(동읍면)별", "행정구역(동읍면)별", "[B]5세별", "2024.11 월"]]
tmp_dat.columns = ["지역코드", "지역명", "연령코드", "인구"]

print(tmp_dat.head(),'\n')
print(tmp_dat.tail(),'\n')
print(tmp_dat.info())

  지역코드 지역명  연령코드        인구
0   00  전국     0  51226757
1   00  전국     5   1272796
2   00  전국    10   1879586
3   00  전국    15   2325120
4   00  전국    20   2295875 

             지역코드  지역명  연령코드   인구
85795  5013062000  예래동    85  140
85796  5013062000  예래동    90  108
85797  5013062000  예래동    95   53
85798  5013062000  예래동   100   11
85799  5013062000  예래동   105    1 

<class 'pandas.DataFrame'>
RangeIndex: 85800 entries, 0 to 85799
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   지역코드    85800 non-null  str  
 1   지역명     85800 non-null  str  
 2   연령코드    85800 non-null  int64
 3   인구      85800 non-null  int64
dtypes: int64(2), str(2)
memory usage: 4.2 MB
None


## 3. 시도, 시군구, 읍면동 구분
- `연령코드`가 `0`이면 전체 연령의 총인구이다.
- 지역코드의 길이는 시도 2자리, 시군구 관련 단위 5자리, 읍면동 10자리이다.
- `00`은 전국이므로 시도 자료에서 제외한다.

In [ ]:
total_dat = tmp_dat.loc[tmp_dat["연령코드"] == 0, :]

total_dat.head()

,지역코드,지역명,연령코드,인구
0,00,전국,0,51226757
22,11,서울특별시,0,9335495
44,11110,종로구,0,138577
66,1111051500,청운효자동,0,11072
88,1111053000,사직동,0,9011


- 지역코드의 길이가 2인 데이터를 `sido_dat`으로 저장한다. 하지만, `Pandas`에서는 `원소별 연산(Vectorization)`을 지원하지만, 모든 Python 함수가 자동으로 원소별 적용되는 것은 아니다.
- 예를 들어, 일반적인 `len()`은 `Series` 전체의 원소 개수를 반환하며, 각 원소의 문자열 길이를 계산하지 않는다.
- 따라서, Pandas의 문자열 접근자(String accessor)인 `.str`을 이용하여 각 원소에 `len()`을 적용한다.
- `.str`의 역할은 Series의 각 원소에 문자열 연산을 적용할 수 있도록 하는 것이다.

In [ ]:
len(total_dat["지역코드"])

3900

In [ ]:
total_dat["지역코드"].str.len()

0         2
22        2
44        5
66       10
88       10
         ..
85690    10
85712    10
85734    10
85756    10
85778    10
Name: 지역코드, Length: 3900, dtype: int64

In [ ]:
total_dat["지역코드"].str.len() == 2

0         True
22        True
44       False
66       False
88       False
         ...  
85690    False
85712    False
85734    False
85756    False
85778    False
Name: 지역코드, Length: 3900, dtype: bool

In [ ]:
sido_dat = total_dat.loc[(total_dat["지역코드"].str.len() == 2) & (total_dat["지역코드"] != "00"), ["지역코드", "지역명"]] # "00" 이면 전국이라 뺀 것
sido_dat.columns = ["시도코드", "시도명"]

sido_dat

,시도코드,시도명
22,11,서울특별시
9966,26,부산광역시
14850,27,대구광역시
18414,28,인천광역시
22176,29,광주광역시
24420,30,대전광역시
26356,31,울산광역시
27698,36,세종특별자치시
28270,41,경기도
42702,51,강원특별자치도


- 지역코드의 길이가 5인 데이터를 `sgg_dat`으로 저장한다.

In [ ]:
sgg_dat = total_dat.loc[total_dat["지역코드"].str.len() == 5, ["지역코드", "지역명"]]
sgg_dat.columns = ["시군구코드", "시군구명"]

sgg_dat

,시군구코드,시군구명
44,11110,종로구
440,11140,중구
792,11170,용산구
1166,11200,성동구
1562,11215,광진구
...,...,...
83842,48870,함양군
84106,48880,거창군
84392,48890,합천군
84810,50110,제주시


- 지역코드의 길이가 10인 데이터를 `dong_dat`으로 저장한다.

In [ ]:
dong_dat = total_dat.loc[total_dat["지역코드"].str.len() == 10, ["지역코드", "지역명", "인구"]]
dong_dat.columns = ["읍면동코드", "읍면동명", "총인구"]

dong_dat

,읍면동코드,읍면동명,총인구
66,1111051500,청운효자동,11072
88,1111053000,사직동,9011
110,1111054000,삼청동,2220
132,1111055000,부암동,9045
154,1111056000,평창동,17239
...,...,...,...
85690,5013058000,서홍동,11134
85712,5013059000,대륜동,15097
85734,5013060000,대천동,13712
85756,5013061000,중문동,12252


## 4. 시와 시군구 구분
- 5자리 지역코드에는 `수원시(41110)`와 `장안구(41111)`처럼 상위 시와 하위 일반구가 함께 존재한다. 이를 구분하기 위해 si와 sgg 변수를 생성한다.
- 일반구가 있는 지역에서는 si에 상위 시 이름을, sgg에 일반구 이름을 저장한다. 일반구가 없는 지역에서는 si와 sgg에 동일한 지역명을 저장한다.
- 먼저 지역명의 마지막 글자가 `시`인 행을 선택하여 `si_dat`을 생성한다. 이때 `.str[-1]`을 이용한다.

In [ ]:
si_dat = sgg_dat.loc[sgg_dat["시군구명"].str[-1] == "시", ["시군구코드", "시군구명"]] # ["시군구명"].str[-1]: 마지막 단어 가져옴
si_dat.columns = ["시코드", "si"]

si_dat  # 마지막 단어가 시인 것만 가져옴

,시코드,si
27720,36110,세종특별자치시
28314,41110,수원시
29392,41130,성남시
30580,41150,의정부시
30932,41170,안양시
...,...,...
80630,48270,밀양시
81026,48310,거제시
81510,48330,양산시
84810,50110,제주시


- `수원시(41110)`와 `장안구(41111)`는 모두 5자리 지역코드이지만, 앞 4자리가 동일하다.
- 이 자료에서는 일반구가 속한 상위 시의 코드가 앞 4자리와 마지막 숫자 0으로 구성된다. 따라서 장안구 코드의 앞 4자리에 0을 붙이면 수원시 코드인 41110을 얻을 수 있다.
- 생성한 시코드를 기준으로 `si_dat`과 `left join`을 수행한다.
- 상위 시코드와 일치하는 데이터가 없으면 `si`에 결측값이 발생하므로, `fillna()`를 이용하여 `sgg`의 값으로 대체한다.

In [ ]:
sgg_dat["시코드"] = sgg_dat["시군구코드"].str[:4] + "0"
sgg_dat["sgg"] = sgg_dat["시군구명"]

sgg_dat

,시군구코드,시군구명,시코드,sgg
44,11110,종로구,11110,종로구
440,11140,중구,11140,중구
792,11170,용산구,11170,용산구
1166,11200,성동구,11200,성동구
1562,11215,광진구,11210,광진구
...,...,...,...,...
83842,48870,함양군,48870,함양군
84106,48880,거창군,48880,거창군
84392,48890,합천군,48890,합천군
84810,50110,제주시,50110,제주시


- 41111 => 41110

In [ ]:
si_dat

,시코드,si
27720,36110,세종특별자치시
28314,41110,수원시
29392,41130,성남시
30580,41150,의정부시
30932,41170,안양시
...,...,...
80630,48270,밀양시
81026,48310,거제시
81510,48330,양산시
84810,50110,제주시


In [ ]:
sgg_dat

,시군구코드,시군구명,시코드,sgg
44,11110,종로구,11110,종로구
440,11140,중구,11140,중구
792,11170,용산구,11170,용산구
1166,11200,성동구,11200,성동구
1562,11215,광진구,11210,광진구
...,...,...,...,...
83842,48870,함양군,48870,함양군
84106,48880,거창군,48880,거창군
84392,48890,합천군,48890,합천군
84810,50110,제주시,50110,제주시


In [ ]:
sgg_dat["시코드"] = sgg_dat["시군구코드"].str[:4] + "0"
sgg_dat["sgg"] = sgg_dat["시군구명"]

sgg_dat = pd.merge(sgg_dat, si_dat, on = "시코드", how = "left")
sgg_dat["si"] = sgg_dat["si"].fillna(sgg_dat["sgg"])

sgg_dat

,시군구코드,시군구명,시코드,sgg,si
0,11110,종로구,11110,종로구,종로구
1,11140,중구,11140,중구,중구
2,11170,용산구,11170,용산구,용산구
3,11200,성동구,11200,성동구,성동구
4,11215,광진구,11210,광진구,광진구
...,...,...,...,...,...
261,48870,함양군,48870,함양군,함양군
262,48880,거창군,48880,거창군,거창군
263,48890,합천군,48890,합천군,합천군
264,50110,제주시,50110,제주시,제주시


- 경기도 장안구는 `si = 수원시`, `sgg = 장안구`가 되며, 서울특별시 강남구는 `si = 강남구`, `sgg = 강남구`가 된다.

In [ ]:
sgg_dat.loc[sgg_dat["시군구코드"].isin(["41111", "11680"]), ["시군구코드", "si", "sgg"]]

,시군구코드,si,sgg
22,11680,강남구,강남구
78,41111,수원시,장안구


## 5. 행정구역 코드로 merge 결합
- 시군구코드의 앞 2자리는 시도코드이다.
- 읍면동코드의 앞 5자리는 해당 읍면동의 시군구코드이다.
- `how = "left"`를 이용하여 자료를 연결한다.

In [ ]:
sgg_dat["시도코드"] = sgg_dat["시군구코드"].str[:2]  # **중요**
dong_dat["시군구코드"] = dong_dat["읍면동코드"].str[:5]

In [ ]:
sgg_dat

,시군구코드,시군구명,시코드,sgg,si,시도코드
0,11110,종로구,11110,종로구,종로구,11
1,11140,중구,11140,중구,중구,11
2,11170,용산구,11170,용산구,용산구,11
3,11200,성동구,11200,성동구,성동구,11
4,11215,광진구,11210,광진구,광진구,11
...,...,...,...,...,...,...
261,48870,함양군,48870,함양군,함양군,48
262,48880,거창군,48880,거창군,거창군,48
263,48890,합천군,48890,합천군,합천군,48
264,50110,제주시,50110,제주시,제주시,50


In [ ]:
dong_dat

,읍면동코드,읍면동명,총인구,시군구코드
66,1111051500,청운효자동,11072,11110
88,1111053000,사직동,9011,11110
110,1111054000,삼청동,2220,11110
132,1111055000,부암동,9045,11110
154,1111056000,평창동,17239,11110
...,...,...,...,...
85690,5013058000,서홍동,11134,50130
85712,5013059000,대륜동,15097,50130
85734,5013060000,대천동,13712,50130
85756,5013061000,중문동,12252,50130


- 먼저 시도와 시군구 정보를 결합한다.

In [ ]:
join_dat = pd.merge(sido_dat, sgg_dat, on = "시도코드", how = "left")

join_dat

,시도코드,시도명,시군구코드,시군구명,시코드,sgg,si
0,11,서울특별시,11110,종로구,11110,종로구,종로구
1,11,서울특별시,11140,중구,11140,중구,중구
2,11,서울특별시,11170,용산구,11170,용산구,용산구
3,11,서울특별시,11200,성동구,11200,성동구,성동구
4,11,서울특별시,11215,광진구,11210,광진구,광진구
...,...,...,...,...,...,...,...
261,48,경상남도,48870,함양군,48870,함양군,함양군
262,48,경상남도,48880,거창군,48880,거창군,거창군
263,48,경상남도,48890,합천군,48890,합천군,합천군
264,50,제주특별자치도,50110,제주시,50110,제주시,제주시


- 읍면동 정보를 추가한다.
- `수원시`처럼 일반구가 있는 경우, 상위 시의 5자리 코드에는 읍면동 데이터가 직접 연결되지 않는다.
- 그렇기에, `left join` 이후 읍면동코드가 결측값인 상위 지역 행을 제거하여 읍면동 단위의 데이터만 남긴다.

In [ ]:
join_dat = pd.merge(join_dat, dong_dat, on = "시군구코드", how = "left") # 시, 도가 기준
join_dat = join_dat.dropna(subset = ["읍면동코드"])

join_dat

,시도코드,시도명,시군구코드,시군구명,시코드,sgg,si,읍면동코드,읍면동명,총인구
0,11,서울특별시,11110,종로구,11110,종로구,종로구,1111051500,청운효자동,11072.0
1,11,서울특별시,11110,종로구,11110,종로구,종로구,1111053000,사직동,9011.0
2,11,서울특별시,11110,종로구,11110,종로구,종로구,1111054000,삼청동,2220.0
3,11,서울특별시,11110,종로구,11110,종로구,종로구,1111055000,부암동,9045.0
4,11,서울특별시,11110,종로구,11110,종로구,종로구,1111056000,평창동,17239.0
...,...,...,...,...,...,...,...,...,...,...
3625,50,제주특별자치도,50130,서귀포시,50130,서귀포시,서귀포시,5013058000,서홍동,11134.0
3626,50,제주특별자치도,50130,서귀포시,50130,서귀포시,서귀포시,5013059000,대륜동,15097.0
3627,50,제주특별자치도,50130,서귀포시,50130,서귀포시,서귀포시,5013060000,대천동,13712.0
3628,50,제주특별자치도,50130,서귀포시,50130,서귀포시,서귀포시,5013061000,중문동,12252.0


In [ ]:
join_dat.loc[join_dat["시군구코드"].isin(["41111"]), :].head()

,시도코드,시도명,시군구코드,시군구명,시코드,sgg,si,읍면동코드,읍면동명,총인구
1202,41,경기도,41111,장안구,41110,장안구,수원시,4111156000,파장동,24781.0
1203,41,경기도,41111,장안구,41110,장안구,수원시,4111156600,율천동,39888.0
1204,41,경기도,41111,장안구,41110,장안구,수원시,4111157100,정자1동,33427.0
1205,41,경기도,41111,장안구,41110,장안구,수원시,4111157200,정자2동,38549.0
1206,41,경기도,41111,장안구,41110,장안구,수원시,4111157300,정자3동,39018.0


In [ ]:
join_dat.loc[join_dat["시군구코드"].isin(["41115"]), :].head()

,시도코드,시도명,시군구코드,시군구명,시코드,sgg,si,읍면동코드,읍면동명,총인구
1224,41,경기도,41115,팔달구,41110,팔달구,수원시,4111565000,매교동,21634.0
1225,41,경기도,41115,팔달구,41110,팔달구,수원시,4111566000,매산동,9832.0
1226,41,경기도,41115,팔달구,41110,팔달구,수원시,4111567000,고등동,19194.0
1227,41,경기도,41115,팔달구,41110,팔달구,수원시,4111568000,화서1동,21000.0
1228,41,경기도,41115,팔달구,41110,팔달구,수원시,4111569000,화서2동,24252.0


## 6. groupby를 이용한 인구 계산
- 먼저 65세 이상 고령인구를 생성한다.
- 원본에서 `연령코드`가 `70`이면 `65 - 69세`이므로, `70` 이상을 합산한다.
- 읍면동별 연령 자료만 선택하여 시도·시군구의 인구가 중복되지 않도록 한다.

In [ ]:
old_dat = tmp_dat.loc[(tmp_dat["지역코드"].str.len() == 10) & (tmp_dat["연령코드"] >= 70), :]
old_dat

,지역코드,지역명,연령코드,인구
80,1111051500,청운효자동,70,664
81,1111051500,청운효자동,75,483
82,1111051500,청운효자동,80,423
83,1111051500,청운효자동,85,346
84,1111051500,청운효자동,90,215
...,...,...,...,...
85795,5013062000,예래동,85,140
85796,5013062000,예래동,90,108
85797,5013062000,예래동,95,53
85798,5013062000,예래동,100,11


In [ ]:
old_dat = tmp_dat.loc[(tmp_dat["지역코드"].str.len() == 10) & (tmp_dat["연령코드"] >= 70), :]

old_dat = old_dat.groupby("지역코드")["인구"].sum().reset_index()
old_dat = old_dat.rename(columns = {"인구": "고령인구"})

old_dat

,지역코드,고령인구
0,1111051500,2226
1,1111053000,1932
2,1111054000,622
3,1111055000,1976
4,1111056000,3787
...,...,...
3611,5280038000,1227
3612,5280039000,994
3613,5280040000,1220
3614,5280041000,1129


- `old_dat`에는 읍면동별 65세 이상 고령인구만 존재한다. 따라서 앞에서 생성한 `join_dat`과 결합하여 총인구와 고령인구를 하나의 DataFrame으로 만든다.

In [ ]:
join_dat = pd.merge(join_dat, old_dat, left_on = "읍면동코드", right_on = "지역코드", how = "left")
join_dat

,시도코드,시도명,시군구코드,시군구명,시코드,sgg,si,읍면동코드,읍면동명,총인구,지역코드,고령인구
0,11,서울특별시,11110,종로구,11110,종로구,종로구,1111051500,청운효자동,11072.0,1111051500,2226
1,11,서울특별시,11110,종로구,11110,종로구,종로구,1111053000,사직동,9011.0,1111053000,1932
2,11,서울특별시,11110,종로구,11110,종로구,종로구,1111054000,삼청동,2220.0,1111054000,622
3,11,서울특별시,11110,종로구,11110,종로구,종로구,1111055000,부암동,9045.0,1111055000,1976
4,11,서울특별시,11110,종로구,11110,종로구,종로구,1111056000,평창동,17239.0,1111056000,3787
...,...,...,...,...,...,...,...,...,...,...,...,...
3611,50,제주특별자치도,50130,서귀포시,50130,서귀포시,서귀포시,5013058000,서홍동,11134.0,5013058000,1803
3612,50,제주특별자치도,50130,서귀포시,50130,서귀포시,서귀포시,5013059000,대륜동,15097.0,5013059000,2874
3613,50,제주특별자치도,50130,서귀포시,50130,서귀포시,서귀포시,5013060000,대천동,13712.0,5013060000,2423
3614,50,제주특별자치도,50130,서귀포시,50130,서귀포시,서귀포시,5013061000,중문동,12252.0,5013061000,2289


## 7. Simple regression
- 회귀분석은 시군구별 집계자료가 아니라 **읍면동별 자료(`join_dat`)**로 수행한다.
- $x$: 읍면동의 총인구수(명), $y$: 65세 이상 고령인구수(명).
- 절편을 포함한 단순선형회귀의 최소제곱법(OLS) 추정량은 다음과 같다.
$$\hat{\beta}_1 = \frac{\sum_i (x_i-\bar{x})(y_i-\bar{y})}{\sum_i (x_i-\bar{x})^2}$$
$$\hat{\beta}_0 = \bar{y} - \hat{\beta}_1\bar{x}$$
- 먼저 수식을 이용하여 절편과 기울기를 직접 계산한다.

In [ ]:
x = join_dat["총인구"].to_numpy()
y = join_dat["고령인구"].to_numpy()

x_mean = np.mean(x)
y_mean = np.mean(y)

beta_1 = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)
beta_0 = y_mean - beta_1 * x_mean

print(beta_1, beta_0)

0.12366346735736533 1068.6741172308666


- 같은 자료를 `LinearRegression`에 적용하여 직접 계산한 결과와 비교한다.
- `scikit-learn`에서는 설명변수 `X`를 2차원 배열로 입력하므로 `reshape(-1, 1)`을 사용한다.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x.reshape(-1, 1), y)

compare_dat = pd.DataFrame({
    "직접 계산": [beta_0, beta_1],
    "패키지": [model.intercept_, model.coef_[0]]
})

compare_dat

,직접 계산,패키지
0,1068.674117,1068.674117
1,0.123663,0.123663
